# RoadSage Phase 1 — Data Exploration & Augmentation Validation

This notebook provides an end-to-end walkthrough of:
1. Loading and inspecting the raw MNNIT campus road image dataset
2. Running the quality filter pipeline and visualising rejection reasons
3. Validating the augmentation pipeline on sample images
4. Reviewing verified images with quality overlays

> **Environment**: Run from the project root (`Road-Sage/`) or from `notebooks/` — the setup cell handles both.

In [ ]:
# ── Environment setup ────────────────────────────────────────────────────────
import os
import sys
from pathlib import Path

_here = Path(os.getcwd())
if _here.name == 'notebooks':
    _project_root = str(_here.parent)
    os.chdir(_project_root)
else:
    _project_root = str(_here)

if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

print(f'Project root: {_project_root}')

import random
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

warnings.filterwarnings('ignore')

from app.preprocessing.augmentation import RoadSageAugmentor
from app.preprocessing.image_quality import (
    QualityChecker,
    compute_image_stats,
    visualize_quality_check,
)
from training.scripts.filter_dataset import run_filter_pipeline

print('All imports OK.')

In [ ]:
# ── Load configuration ────────────────────────────────────────────────────────
CONFIG_PATH = 'configs/production.yaml'

with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

aug_cfg   = config.get('augmentation', {})
dec_cfg   = config.get('decision_engine', {})
model_cfg = config.get('model', {})

print('=== Key configuration settings ===')
print(f"  Project environment : {config['project']['environment']}")
print(f"  Input resolution    : {model_cfg.get('input_height')} x {model_cfg.get('input_width')}")
print(f"  CLAHE probability   : {aug_cfg.get('clahe_prob')}")
print(f"  Horizontal flip     : {aug_cfg.get('horizontal_flip_prob')}")
print(f"  Min confidence      : {dec_cfg.get('min_confidence')}")
print(f"  Obstacle stop dist  : {dec_cfg.get('obstacle_stop_distance')} m")

## Dataset Overview

Scan the raw `rgb/` directory, compute per-image statistics, and plot brightness and blur score distributions.

In [ ]:
# ── Dataset scan ──────────────────────────────────────────────────────────────
RAW_DIR    = Path('rgb')
VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

image_paths = [
    p for p in sorted(RAW_DIR.rglob('*'))
    if p.suffix.lower() in VALID_EXTS
] if RAW_DIR.exists() else []

print(f"Images found in '{RAW_DIR}': {len(image_paths)}")

if not image_paths:
    print('WARNING: no images found — skipping stats computation.')
else:
    brightness_scores, blur_scores = [], []
    checker = QualityChecker()

    for path in image_paths:
        try:
            result = checker.check_from_path(str(path))
            brightness_scores.append(result.brightness_score)
            blur_scores.append(result.blur_score)
        except Exception as exc:
            print(f'  Skipping {path.name}: {exc}')

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].hist(brightness_scores, bins=30, color='steelblue', edgecolor='white')
    axes[0].axvline(30,  color='red',    linestyle='--', label='min (30)')
    axes[0].axvline(220, color='orange', linestyle='--', label='max (220)')
    axes[0].set_title('Brightness Score Distribution')
    axes[0].set_xlabel('Mean pixel value')
    axes[0].set_ylabel('Count')
    axes[0].legend()

    axes[1].hist(blur_scores, bins=30, color='seagreen', edgecolor='white')
    axes[1].axvline(50, color='red', linestyle='--', label='threshold (50)')
    axes[1].set_title('Blur Score Distribution (Laplacian variance)')
    axes[1].set_xlabel('Variance')
    axes[1].set_ylabel('Count')
    axes[1].legend()

    plt.tight_layout()
    plt.show()
    print(f'Brightness  mean: {np.mean(brightness_scores):.1f}, std: {np.std(brightness_scores):.1f}')
    print(f'Blur        mean: {np.mean(blur_scores):.1f},       std: {np.std(blur_scores):.1f}')

## Quality Filter Results

Run the full 4-gate filter pipeline (blur → brightness → road coverage → duplicate) and visualise how many images are rejected at each gate.

In [ ]:
# ── Run filter pipeline ───────────────────────────────────────────────────────
VERIFIED_DIR  = 'data/mnnit/verified'
filter_config = {
    'blur_threshold'      : 50.0,
    'brightness_min'      : 30.0,
    'brightness_max'      : 220.0,
    'road_coverage_min'   : 0.20,
    'duplicate_threshold' : 0.98,
}

if not RAW_DIR.exists() or not image_paths:
    print('WARNING: no source images — skipping pipeline run.')
    stats = None
else:
    stats = run_filter_pipeline(
        source_dir=str(RAW_DIR),
        output_dir=VERIFIED_DIR,
        config=filter_config,
    )

    print('=== Filter pipeline results ===')
    for k, v in stats.items():
        print(f'  {k:<30}: {v}')

    rejection_keys   = ['rejected_blur', 'rejected_brightness', 'rejected_road_coverage', 'rejected_duplicate']
    rejection_labels = [k.replace('rejected_', '').replace('_', '\n') for k in rejection_keys]
    rejection_counts  = [stats.get(k, 0) for k in rejection_keys]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(rejection_labels, rejection_counts, color=['#e74c3c', '#e67e22', '#8e44ad', '#2980b9'])
    ax.bar_label(bars, padding=4)
    ax.set_title(
        f"Rejection Breakdown  (total: {stats.get('total_processed', 0)}, "
        f"accepted: {stats.get('passed', 0)}, "
        f"rate: {stats.get('acceptance_rate', 0):.1%})"
    )
    ax.set_ylabel('Images rejected')
    plt.tight_layout()
    plt.show()

## Augmentation Validation

Apply the training augmentation pipeline to 5 randomly sampled images and show 3 augmented variants of each in a 5 x 4 grid (original + 3 augmented).

In [ ]:
# ── Augmentation preview ──────────────────────────────────────────────────────
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

verified_paths = [
    p for p in sorted(Path(VERIFIED_DIR).rglob('*'))
    if p.suffix.lower() in VALID_EXTS
] if Path(VERIFIED_DIR).exists() else []

pool = verified_paths if verified_paths else image_paths

if not pool:
    print('WARNING: no images available for augmentation preview.')
else:
    augmentor    = RoadSageAugmentor(config.get('augmentation', {}))
    sample_paths = random.sample(pool, min(5, len(pool)))

    _MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    _STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def _to_display(img):
        if img.dtype in (np.float32, np.float64):
            img = np.clip(img * _STD + _MEAN, 0.0, 1.0)
            img = (img * 255).astype(np.uint8)
        if img.ndim == 3 and img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return img

    n_rows, n_cols = len(sample_paths), 4
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 10))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for col, title in enumerate(['Original', 'Aug #1', 'Aug #2', 'Aug #3']):
        axes[0, col].set_title(title, fontsize=11, fontweight='bold')

    for row, img_path in enumerate(sample_paths):
        original = cv2.imread(str(img_path))
        if original is None:
            continue
        axes[row, 0].imshow(_to_display(original))
        axes[row, 0].set_ylabel(img_path.name[:20], fontsize=8)
        for col in range(1, 4):
            aug_img = augmentor.augment(original.copy())
            axes[row, col].imshow(_to_display(aug_img))
        for ax in axes[row]:
            ax.axis('off')

    plt.suptitle('Augmentation Validation — RoadSage Training Pipeline', fontsize=13, y=1.01)
    plt.tight_layout()
    save_path = OUTPUT_DIR / 'augmentation_preview.png'
    fig.savefig(str(save_path), dpi=120, bbox_inches='tight')
    print(f'Saved augmentation preview -> {save_path}')
    plt.show()

## Sample Visualizations

Display 10 randomly selected verified images with the quality-check overlay (green border = pass, red border = fail).

In [ ]:
# ── Quality overlay grid ──────────────────────────────────────────────────────
display_pool = verified_paths if verified_paths else image_paths

if not display_pool:
    print('WARNING: no images available for quality overlay display.')
else:
    sample_display = random.sample(display_pool, min(10, len(display_pool)))
    n_rows, n_cols = 2, 5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 7))
    checker = QualityChecker()

    for idx, img_path in enumerate(sample_display):
        row, col = divmod(idx, n_cols)
        ax = axes[row, col]
        image = cv2.imread(str(img_path))
        if image is None:
            ax.axis('off')
            continue
        result      = checker.check(image)
        overlay     = visualize_quality_check(image, result)
        overlay_rgb = cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB)
        status = 'PASS' if result.passed else f'FAIL ({result.rejection_reason})'
        ax.imshow(overlay_rgb)
        ax.set_title(f'{img_path.name[:18]}\n{status}', fontsize=7)
        ax.axis('off')

    for idx in range(len(sample_display), n_rows * n_cols):
        row, col = divmod(idx, n_cols)
        axes[row, col].axis('off')

    plt.suptitle('Quality Check Overlays — Verified Dataset Sample', fontsize=13)
    plt.tight_layout()
    plt.show()

## Summary

### What this notebook covers (Phase 1)

| Section | Output |
|---------|--------|
| Dataset overview | Brightness & blur histograms for raw images |
| Quality filter | Rejection breakdown bar chart |
| Augmentation validation | 5 x 4 grid saved to `outputs/augmentation_preview.png` |
| Sample visualizations | 2 x 5 quality-overlay grid |

### Manual checks before Phase 2

1. **Brightness histogram** — The bulk of images should fall between 30 and 220. A spike near 0 suggests under-exposed frames; raise `brightness_min` or add gamma correction.
2. **Blur distribution** — If `rejected_blur` exceeds 30 % of total, consider loosening the threshold or adding sharpening in preprocessing.
3. **Acceptance rate** — Target >= 70 %. Lower rates indicate collection issues (camera shake, lens fog).
4. **Augmentation grid** — Confirm horizontal flips produce plausible road geometry. Check that rain/shadow overlays look realistic for MNNIT conditions.
5. **Quality overlays** — Green-bordered images should be usable for training; red-bordered ones should show an obvious quality problem.

Once these checks pass, proceed to **Phase 2: Lane Detection Model Integration**.